## Section 0: Configuration & Constants

In [8]:
# ============================================================================
# SECTION 0: CONFIGURATION & CONSTANTS
# ============================================================================

RANDOM_SEED = 231

NUM_CLASSES = 4
PARTICLE_NAMES = ['Pion', 'Kaon', 'Proton', 'Electron']
PDG_TO_SPECIES = {
    211: 0,
    321: 1,
    2212: 2,
    11: 3,
}

MOMENTUM_RANGES = [
    {'key': 'full', 'name': 'Full Spectrum (0.1-∞ GeV/c)', 'min': 0.1, 'max': float('inf')},
    {'key': '0.7-1.5', 'name': '0.7-1.5 GeV/c', 'min': 0.7, 'max': 1.5},
    {'key': '1-3', 'name': '1-3 GeV/c', 'min': 1.0, 'max': 3.0},
]

CSV_PATH = '/kaggle/input/datasets/robertforynski/new-ao2d-lhc25f60544122/pid_features_*.csv'

TRAINING_FEATURES = [
    'pt', 'eta', 'phi',
    'tpc_signal', 'tpc_nsigma_pi', 'tpc_nsigma_ka', 'tpc_nsigma_pr', 'tpc_nsigma_el',
    'tof_beta', 'tof_nsigma_pi', 'tof_nsigma_ka', 'tof_nsigma_pr', 'tof_nsigma_el',
    'bayes_prob_pi', 'bayes_prob_ka', 'bayes_prob_pr', 'bayes_prob_el',
    'dca_xy', 'dca_z',
    'has_tpc', 'has_tof',
]

DETECTOR_GROUPS = {
    'tpc': ['tpc_signal', 'tpc_nsigma_pi', 'tpc_nsigma_ka', 'tpc_nsigma_pr', 'tpc_nsigma_el'],
    'tof': ['tof_beta', 'tof_nsigma_pi', 'tof_nsigma_ka', 'tof_nsigma_pr', 'tof_nsigma_el'],
    'bayes': ['bayes_prob_pi', 'bayes_prob_ka', 'bayes_prob_pr', 'bayes_prob_el'],
    'kinematics': ['pt', 'eta', 'phi', 'dca_xy', 'dca_z'],
}

MODEL_TYPES = [
    'JAX_SimpleNN',
    'JAX_DNN',
    'JAX_FSE_Attention',
    'JAX_FSE_Attention_DetectorAware',
]

HYPERPARAMETERS = {
    'JAX_SimpleNN': {
        'hidden_dims': [512, 256, 128, 64],
        'dropout_rate': 0.5,
        'learning_rate': 1e-4,
        'batch_size': 256,
        'num_epochs': 100,
        'patience': 30,
    },
    'JAX_DNN': {
        'hidden_dims': [1024, 512, 256, 128, 64],
        'dropout_rate': 0.5,
        'learning_rate': 5e-5,
        'batch_size': 256,
        'num_epochs': 100,
        'patience': 30,
    },
    'JAX_FSE_Attention': {
        'hidden_dim': 64,
        'num_heads': 4,
        'dropout_rate': 0.5,
        'learning_rate': 1e-4,
        'batch_size': 256,
        'num_epochs': 100,
        'patience': 30,
    },
    'JAX_FSE_Attention_DetectorAware': {
        'hidden_dim': 64,
        'num_heads': 4,
        'dropout_rate': 0.5,
        'learning_rate': 1e-4,
        'batch_size': 256,
        'num_epochs': 100,
        'patience': 30,
        'detector_embed_dim': 8,
    },
}

FORCE_TRAINING = {
    'JAX_SimpleNN': {'full': False, '0.7-1.5': False, '1-3': False},
    'JAX_DNN': {'full': False, '0.7-1.5': False, '1-3': False},
    'JAX_FSE_Attention': {'full': False, '0.7-1.5': False, '1-3': False},
    'JAX_FSE_Attention_DetectorAware': {'full': False, '0.7-1.5': False, '1-3': False},
}

model_colors = {
    'JAX_SimpleNN': '#3B82F6',           # blue
    'JAX_DNN': '#F59E0B',               # amber
    'JAX_FSE_Attention': '#22C55E',     # green
    'JAX_FSE_Attention_DetectorAware': '#EF4444',  # red
    'Bayesian_PID': '#F97316',          # orange
}

print("✓ Configuration loaded")
print(f"  Momentum ranges: {len(MOMENTUM_RANGES)}")
print(f"  Model types: {len(MODEL_TYPES)}")
print(f"  Particle classes: {NUM_CLASSES}")
print(f"  Training features: {len(TRAINING_FEATURES)}\n")


Config ready.


## Section 1: Imports

In [ ]:
# ============================================================================
# SECTION 1: IMPORTS
# ============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import json
import os
import time
import warnings
from tqdm import tqdm
warnings.filterwarnings('ignore')

import jax
import jax.numpy as jnp
from jax import random, grad, jit, vmap
import optax
from flax import linen as nn
from flax.training import train_state

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, roc_curve, auc, roc_auc_score, classification_report, accuracy_score
from sklearn.utils.class_weight import compute_class_weight

print(f"✓ JAX version: {jax.__version__}")
print(f"✓ Available devices: {jax.devices()}")
print(f"✓ All libraries imported\n")

print(f"{'='*80}")
print("✓ SECTION 1 COMPLETE")
print(f"{'='*80}\n")


## Section 2: Data Loading & Preprocessing Utilities

In [ ]:
# ============================================================================
# 2.1: DATA LOADING & PREPROCESSING
# ============================================================================

def load_data(csv_path):
    """Load CSV data in chunks for memory efficiency."""
    print(f"Loading data from {csv_path}...")
    df_iter = pd.read_csv(csv_path, dtype='float32', chunksize=500000, low_memory=False)
    df = pd.concat(df_iter, ignore_index=True)
    print(f"✓ Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns\n")
    return df


def pdg_to_species(pdg):
    """Convert PDG code to species index."""
    ap = abs(int(pdg))
    return PDG_TO_SPECIES.get(ap, -1)


def preprocess_momentum_range(df, momentum_range):

    print(f"\n{'─'*80}")
    print(f"Preprocessing: {momentum_range['name']}")
    print(f"{'─'*80}")
    
    # Filter by momentum
    df_filtered = df[(df['p'] >= momentum_range['min']) & 
                     (df['p'] < momentum_range['max'])].copy()
    print(f"  Samples after momentum filter: {len(df_filtered):,}")
    
    bayes_features = DETECTOR_GROUPS['bayes']
    
    # Missing value analysis
    print("\n  Missing value analysis:")
    missing_before = df_filtered[TRAINING_FEATURES].isnull().sum()
    total_missing = missing_before.sum()
    
    if total_missing > 0:
        print(f"    Total missing: {total_missing:,}")
    
    # Fill TOF features
    for feat in DETECTOR_GROUPS['tof']:
        if feat in df_filtered.columns:
            fill_val = 0.0 if feat == 'tof_beta' else 999.0
            df_filtered[feat].fillna(fill_val, inplace=True)
    
    # Fill TPC features
    for feat in DETECTOR_GROUPS['tpc']:
        if feat in df_filtered.columns:
            fill_val = 0.0 if feat == 'tpc_signal' else 999.0
            df_filtered[feat].fillna(fill_val, inplace=True)
    
    # ========================================================================
    # CRITICAL: SAVE BAYESIAN MASK BEFORE FILLING
    # ========================================================================
    # Check which tracks have all 4 Bayesian features non-zero BEFORE filling
    bayes_complete_before_fill = (df_filtered[bayes_features] != 0).all(axis=1)
    bayes_pred_before_fill = np.argmax(df_filtered[bayes_features].values, axis=1)
    
    n_real_bayes_before = np.sum(bayes_complete_before_fill)
    pct_real_before = (n_real_bayes_before / len(bayes_complete_before_fill)) * 100
    
    print(f"\n  Bayesian PID Status (BEFORE FILLING):")
    print(f"    REAL Bayesian:   {n_real_bayes_before:,} ({pct_real_before:.2f}%)")
    print(f"    MISSING:         {len(bayes_complete_before_fill) - n_real_bayes_before:,} ({100-pct_real_before:.2f}%)")
    
    # NOW fill Bayesian features (after saving mask)
    for feat in bayes_features:
        if feat in df_filtered.columns:
            df_filtered[feat].fillna(0.25, inplace=True)  # Uniform probability
    
    # Fill kinematic features
    for feat in DETECTOR_GROUPS['kinematics']:
        if feat in df_filtered.columns:
            median_val = df_filtered[feat].median()
            df_filtered[feat].fillna(median_val, inplace=True)
    
    df_filtered['has_tpc'].fillna(0, inplace=True)
    df_filtered['has_tof'].fillna(0, inplace=True)
    
    # Drop any remaining NaN
    if df_filtered[TRAINING_FEATURES].isnull().sum().sum() > 0:
        df_filtered.dropna(subset=TRAINING_FEATURES, inplace=True)
    
    # Create detector group masks
    group_names = list(DETECTOR_GROUPS.keys())
    group_masks_data = []
    
    for g in group_names:
        if g == 'tpc':
            group_masks_data.append(df_filtered['has_tpc'].values.astype('float32'))
        elif g == 'tof':
            group_masks_data.append(df_filtered['has_tof'].values.astype('float32'))
        else:
            group_masks_data.append(np.ones(len(df_filtered), dtype='float32'))
    
    group_masks = np.stack(group_masks_data, axis=1)
    
    # Extract features and labels
    X = df_filtered[TRAINING_FEATURES].values
    y = df_filtered['mc_pdg'].values
    y = np.array([pdg_to_species(pdg) for pdg in y])
    valid_mask = y >= 0
    
    # ========================================================================
    # APPLY VALID_MASK TO BAYESIAN MASKS (saved before filling)
    # ========================================================================
    bayes_availability_mask = bayes_complete_before_fill[valid_mask].astype('float32')
    bayes_pred_original = bayes_pred_before_fill[valid_mask]
    
    n_real_bayes = np.sum(bayes_availability_mask)
    n_filled_bayes = len(bayes_availability_mask) - n_real_bayes
    pct_real = (n_real_bayes / len(bayes_availability_mask)) * 100
    
    print(f"\n  Bayesian PID Status (AFTER filtering valid particles):")
    print(f"    REAL Bayesian:   {n_real_bayes:,} ({pct_real:.2f}%)")
    print(f"    FILLED Bayesian: {n_filled_bayes:,} ({100-pct_real:.2f}%)")
    
    # Apply valid_mask to all data
    X = X[valid_mask]
    y = y[valid_mask]
    group_masks = group_masks[valid_mask]
    
    print(f"\n  Final dataset: {X.shape}")
    print(f"  Class distribution:")
    for i, particle in enumerate(PARTICLE_NAMES):
        count = np.sum(y == i)
        pct = (count / len(y)) * 100
        print(f"    {particle:10s}: {count:6,} ({pct:5.2f}%)")
    
    # ========================================================================
    # PHASE 1: CREATE DETECTOR MODES (NEW!)
    # ========================================================================
    
    print(f"\n  Creating detector modes for Phase 1...")
    
    # Extract has_tpc and has_tof BEFORE scaling
    has_tpc_idx = TRAINING_FEATURES.index('has_tpc')
    has_tof_idx = TRAINING_FEATURES.index('has_tof')
    
    has_tpc_array = X[:, has_tpc_idx].astype(int)
    has_tof_array = X[:, has_tof_idx].astype(int)
    
    # Create detector mode: 0=NONE, 1=TPC_ONLY, 2=TOF_ONLY, 3=TPC_TOF
    detector_mode = np.zeros(len(X), dtype='int32')
    detector_mode[(has_tpc_array == 1) & (has_tof_array == 0)] = 1  # TPC_ONLY
    detector_mode[(has_tpc_array == 0) & (has_tof_array == 1)] = 2  # TOF_ONLY
    detector_mode[(has_tpc_array == 1) & (has_tof_array == 1)] = 3  # TPC_TOF
    
    # Verify distribution
    unique_modes, mode_counts = np.unique(detector_mode, return_counts=True)
    print(f"\n  Detector mode distribution:")
    mode_names_full = ['NONE', 'TPC_ONLY', 'TOF_ONLY', 'TPC_TOF']
    for mode, count in zip(unique_modes, mode_counts):
        pct = (count / len(detector_mode)) * 100
        print(f"    {mode_names_full[mode]:15s} ({mode}): {count:6,} ({pct:5.2f}%)")
    
    # Train-test split (INCLUDING detector modes)
    X_train, X_test, y_train, y_test, masks_train, masks_test, \
    bayes_mask_train, bayes_mask_test, bayes_pred_train, bayes_pred_test, \
    X_train_modes, X_test_modes = train_test_split(
        X, y, group_masks, bayes_availability_mask, bayes_pred_original, detector_mode,
        test_size=0.2, random_state=RANDOM_SEED, stratify=y
    )
    
    # Scale features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    print(f"\n  Train: {len(X_train):,}, Test: {len(X_test):,}")
    
    # Verify test set Bayesian stats
    n_real_test = np.sum(bayes_mask_test)
    pct_real_test = (n_real_test / len(bayes_mask_test)) * 100
    print(f"\n  Test Set Bayesian Stats:")
    print(f"    REAL: {n_real_test:,} ({pct_real_test:.2f}%)")
    print(f"    FILLED: {len(bayes_mask_test) - n_real_test:,} ({100-pct_real_test:.2f}%)")
    
    # Verify test set detector mode distribution
    unique_modes_test, mode_counts_test = np.unique(X_test_modes, return_counts=True)
    print(f"\n  Test Set Detector Mode Distribution:")
    for mode, count in zip(unique_modes_test, mode_counts_test):
        pct = (count / len(X_test_modes)) * 100
        print(f"    {mode_names_full[mode]:15s}: {count:6,} ({pct:5.2f}%)")
    
    return {
        'X_train_scaled': X_train_scaled,
        'X_test_scaled': X_test_scaled,
        'y_train': y_train,
        'y_test': y_test,
        'scaler': scaler,
        'features': TRAINING_FEATURES,
        'masks_train': masks_train,
        'masks_test': masks_test,
        'group_names': group_names,
        'bayes_availability_test': bayes_mask_test,
        'bayes_pred_original_test': bayes_pred_test,
        'detector_modes_train': X_train_modes,      # NEW - Phase 1
        'detector_modes_test': X_test_modes,        # NEW - Phase 1
    }


# ============================================================================
# 2.2: MODEL PERSISTENCE (LOADING/SAVING)
# ============================================================================

def get_model_path(momentum_range_key, model_type, mode='save'):
    """
    Get file path for model save/load.
    Saves to /kaggle/working/, loads from working (priority) or input (fallback).
    """
    model_subdir = "trained_models"  # Organise models in subfolder
    working_path = f"/kaggle/working/{model_subdir}/{momentum_range_key}_{model_type}.pkl"
    input_path = f"/kaggle/input/jax-models/jax-models/{momentum_range_key}_{model_type}.pkl"
    
    if mode == "save":
        return working_path  # Always save to working directory
    else:  # mode == "load"
        # Try working first, then input
        return working_path if os.path.exists(working_path) else input_path


def load_single_model(momentum_range_key, model_type):
    """Load a single model from disk."""
    path = get_model_path(momentum_range_key, model_type, mode="load")
    
    if os.path.exists(path):
        try:
            with open(path, 'rb') as f:
                results = pickle.load(f)
            print(f"✓ Loaded from: {path}")
            return results, path
        except Exception as e:
            print(f"Error loading {path}: {e}")
    
    return None, path


def save_single_model(momentum_range_key, model_type, results):
    """Save a single model to disk."""
    path = get_model_path(momentum_range_key, model_type, mode="save")
    
    # Create directory if it doesn't exist
    os.makedirs(os.path.dirname(path), exist_ok=True)
    
    try:
        with open(path, 'wb') as f:
            pickle.dump(results, f)
        print(f"✓ Saved to: {path}")
    except Exception as e:
        print(f"Error saving to {path}: {e}")

print("✓ Data loading & preprocessing utilities defined (with Phase 1 detector modes)")
print("✓ Model persistence utilities defined")
print(f"\n{'='*80}")
print("✓ SECTION 2 COMPLETE: Utility functions ready")
print("✓ Detector modes will be created during preprocessing")
print(f"{'='*80}\n")


## Section 3: Model Definitions & Training Functions

In [ ]:
# ============================================================================
# SECTION 3: MODEL DEFINITIONS & TRAINING UTILITIES (SHARED + PHASE 1)
# ============================================================================

# ============================================================================
# 3.1: LOSS FUNCTION
# ============================================================================

def focal_loss(logits, labels, class_weights=None, alpha=0.25, gamma=2.0):
    """
    Focal loss for multi-class classification.
    Focuses training on hard negatives and reduces easy negative weight.
    """
    probs = jax.nn.softmax(logits, axis=-1)
    pt = probs[jnp.arange(labels.shape[0]), labels]
    ce_loss = -jnp.log(pt + 1e-7)
    
    w = class_weights[labels] if class_weights is not None else 1.0
    focal_weight = alpha * (1.0 - pt) ** gamma
    loss = jnp.mean(w * focal_weight * ce_loss)
    return loss


# ============================================================================
# 3.2: MODEL ARCHITECTURES
# ============================================================================

class JAX_SimpleNN(nn.Module):
    """Simple feedforward neural network."""
    hidden_dims: list
    num_classes: int
    dropout_rate: float = 0.3
    
    @nn.compact
    def __call__(self, x, training: bool = False):
        for dim in self.hidden_dims:
            x = nn.Dense(dim)(x)
            x = nn.relu(x)
            x = nn.Dropout(rate=self.dropout_rate, deterministic=not training)(x)
        x = nn.Dense(self.num_classes)(x)
        return x


class JAX_DNN(nn.Module):
    """Deeper neural network with batch normalisation."""
    hidden_dims: list
    num_classes: int
    dropout_rate: float = 0.3
    
    @nn.compact
    def __call__(self, x, training: bool = False):
        for dim in self.hidden_dims:
            x = nn.Dense(dim)(x)
            x = nn.BatchNorm(use_running_average=not training)(x)
            x = nn.relu(x)
            x = nn.Dropout(rate=self.dropout_rate, deterministic=not training)(x)
        x = nn.Dense(self.num_classes)(x)
        return x


class JAX_FSE_Attention(nn.Module):
    """Feature Set Embedding with Multi-Head Attention."""
    hidden_dim: int = 64
    num_heads: int = 4
    num_classes: int = 4
    dropout_rate: float = 0.3
    
    @nn.compact
    def __call__(self, x, group_mask, training: bool = False):
        batch_size = x.shape[0]
        num_groups = group_mask.shape[1]
        
        # Project features to per-group embeddings
        feat_proj = nn.Dense(self.hidden_dim * num_groups)(x)
        feat_proj = feat_proj.reshape(batch_size, num_groups, self.hidden_dim)
        feat_proj = feat_proj * group_mask[:, :, None]
        
        # Self-attention over detector groups
        attn_mask = group_mask[:, None, None, :]
        feat_attn = nn.MultiHeadDotProductAttention(num_heads=self.num_heads)(
            feat_proj, feat_proj, mask=attn_mask
        )
        feat_attn = nn.LayerNorm()(feat_attn)
        
        # Gated fusion
        gates = nn.Dense(self.hidden_dim)(feat_attn)
        gates = nn.sigmoid(gates)
        feat_gated = feat_attn * gates
        
        # Masked pooling
        denom = jnp.clip(jnp.sum(group_mask, axis=1, keepdims=True), a_min=1.0)
        pooled = jnp.sum(feat_gated * group_mask[:, :, None], axis=1) / denom
        
        # Classification head
        x = nn.Dense(128)(pooled)
        x = nn.relu(x)
        x = nn.Dropout(rate=self.dropout_rate, deterministic=not training)(x)
        x = nn.Dense(64)(x)
        x = nn.relu(x)
        x = nn.Dropout(rate=self.dropout_rate, deterministic=not training)(x)
        x = nn.Dense(self.num_classes)(x)
        return x


class JAX_FSE_Attention_DetectorAware(nn.Module):
    """
    Feature Set Embedding with Multi-Head Attention + Detector-Aware Head (Phase 1)
    
    Learns specialised decision boundaries per detector configuration:
    - TPC-only tracks (89% in 0.7-1.5 GeV/c)
    - TOF-only tracks (rare, 8.5%)
    - TPC+TOF tracks (best separation)
    
    """
    hidden_dim: int = 64
    num_heads: int = 4
    num_classes: int = 4
    dropout_rate: float = 0.3
    detector_embed_dim: int = 8
    
    @nn.compact
    def __call__(self, x, group_mask, detector_mode, training: bool = False):
        """
        Args:
            x: (batch, num_features) - scaled features
            group_mask: (batch, num_groups) - detector availability mask
            detector_mode: (batch,) - int32 array, detector mode (0-3)
                0=NONE, 1=TPC_ONLY, 2=TOF_ONLY, 3=TPC_TOF
            training: boolean for dropout
        """
        batch_size = x.shape[0]
        num_groups = group_mask.shape[1]
        
        # ====================================================================
        # STANDARD FSE + ATTENTION (unchanged from JAX_FSE_Attention)
        # ====================================================================
        
        # Project features to per-group embeddings
        feat_proj = nn.Dense(self.hidden_dim * num_groups)(x)
        feat_proj = feat_proj.reshape((batch_size, num_groups, self.hidden_dim))
        feat_proj = feat_proj * group_mask[:, :, None]
        
        # Self-attention over detector groups
        attn_mask = group_mask[:, None, None, :]
        feat_attn = nn.MultiHeadDotProductAttention(
            num_heads=self.num_heads
        )(feat_proj, feat_proj, mask=attn_mask)
        
        feat_attn = nn.LayerNorm()(feat_attn)
        
        # Gated fusion
        gates = nn.Dense(self.hidden_dim)(feat_attn)
        gates = nn.sigmoid(gates)
        feat_gated = feat_attn * gates
        
        # Masked pooling
        denom = jnp.clip(jnp.sum(group_mask, axis=1, keepdims=True), a_min=1.0)
        pooled = jnp.sum(feat_gated * group_mask[:, :, None], axis=1) / denom
        
        # ====================================================================
        # DETECTOR-MODE EMBEDDING (NEW! Phase 1)
        # ====================================================================
        
        # One-hot encode detector mode (0, 1, 2, 3) → (batch, 4)
        detector_onehot = jax.nn.one_hot(detector_mode, num_classes=4)
        
        # Dense embedding layer: 4 → detector_embed_dim (8)
        detector_emb = nn.Dense(self.detector_embed_dim)(detector_onehot)
        detector_emb = nn.relu(detector_emb)
        
        # ====================================================================
        # FUSE ATTENTION POOLED + DETECTOR EMBEDDING
        # ====================================================================
        
        # Concatenate: (batch, hidden_dim) + (batch, detector_embed_dim)
        #            → (batch, hidden_dim + detector_embed_dim)
        x_fused = jnp.concatenate([pooled, detector_emb], axis=-1)
        
        # ====================================================================
        # CLASSIFICATION HEAD (expanded input dim)
        # ====================================================================
        
        x_head = nn.Dense(128)(x_fused)
        x_head = nn.relu(x_head)
        x_head = nn.Dropout(rate=self.dropout_rate, deterministic=not training)(x_head)
        
        x_head = nn.Dense(64)(x_head)
        x_head = nn.relu(x_head)
        x_head = nn.Dropout(rate=self.dropout_rate, deterministic=not training)(x_head)
        
        # Output logits
        logits = nn.Dense(self.num_classes)(x_head)
        
        return logits


# ============================================================================
# 3.3: TRAINING STEP FUNCTIONS
# ============================================================================

@jit
def train_step_simple(state, batch_x, batch_y, rng, class_weights):
    """Training step for SimpleNN (no BatchNorm)."""
    def loss_fn(params):
        logits = state.apply_fn({'params': params}, batch_x, training=True, rngs={'dropout': rng})
        loss = focal_loss(logits, batch_y, class_weights=class_weights)
        return loss
    
    loss, grads = jax.value_and_grad(loss_fn)(state.params)
    state = state.apply_gradients(grads=grads)
    return state, loss


@jit
def train_step_batchnorm(state, batch_x, batch_y, rng, class_weights):
    """Training step for DNN (with BatchNorm)."""
    def loss_fn(params):
        variables = {'params': params, 'batch_stats': state.batch_stats}
        logits, new_model_state = state.apply_fn(
            variables, batch_x, training=True, rngs={'dropout': rng}, mutable=['batch_stats']
        )
        loss = focal_loss(logits, batch_y, class_weights=class_weights)
        return loss, new_model_state
    
    (loss, new_model_state), grads = jax.value_and_grad(loss_fn, has_aux=True)(state.params)
    state = state.apply_gradients(grads=grads)
    state = state.replace(batch_stats=new_model_state['batch_stats'])
    return state, loss


@jit
def train_step_fse(state, batch_x, batch_mask, batch_y, rng, class_weights):
    """Training step for FSE+Attention."""
    def loss_fn(params):
        logits = state.apply_fn({'params': params}, batch_x, batch_mask, training=True, rngs={'dropout': rng})
        loss = focal_loss(logits, batch_y, class_weights=class_weights)
        return loss
    
    loss, grads = jax.value_and_grad(loss_fn)(state.params)
    state = state.apply_gradients(grads=grads)
    return state, loss


@jit
def train_step_fse_aware(state, batch_x, batch_mask, batch_modes, batch_y, rng, class_weights):
    """Training step for Detector-Aware FSE+Attention using Focal Loss (Phase 1)"""
    def loss_fn(params):
        logits = state.apply_fn(
            {'params': params}, batch_x, batch_mask, batch_modes, 
            training=True, rngs={'dropout': rng}
        )
        loss = focal_loss(logits, batch_y, class_weights=class_weights, 
                         alpha=0.25, gamma=2.0)
        return loss
    
    loss, grads = jax.value_and_grad(loss_fn)(state.params)
    state = state.apply_gradients(grads=grads)
    return state, loss


# ============================================================================
# 3.4: EVALUATION FUNCTIONS
# ============================================================================

@jit
def eval_step_simple(state, batch_x, batch_y):
    """Evaluation step for SimpleNN."""
    logits = state.apply_fn({'params': state.params}, batch_x, training=False)
    pred = jnp.argmax(logits, axis=-1)
    accuracy = jnp.mean(pred == batch_y)
    return accuracy, logits


@jit
def eval_step_batchnorm(state, batch_x, batch_y):
    """Evaluation step for DNN with BatchNorm."""
    variables = {'params': state.params, 'batch_stats': state.batch_stats}
    logits = state.apply_fn(variables, batch_x, training=False)
    pred = jnp.argmax(logits, axis=-1)
    accuracy = jnp.mean(pred == batch_y)
    return accuracy, logits


@jit
def eval_step_fse(state, batch_x, batch_mask, batch_y):
    """Evaluation step for FSE+Attention."""
    logits = state.apply_fn({'params': state.params}, batch_x, batch_mask, training=False)
    pred = jnp.argmax(logits, axis=-1)
    accuracy = jnp.mean(pred == batch_y)
    return accuracy, logits


@jit
def eval_step_fse_aware(state, batch_x, batch_mask, batch_modes, batch_y):
    """Evaluation step for Detector-Aware FSE+Attention (Phase 1)"""
    logits = state.apply_fn(
        {'params': state.params}, batch_x, batch_mask, batch_modes, training=False
    )
    pred = jnp.argmax(logits, axis=-1)
    accuracy = jnp.mean(pred == batch_y)
    return accuracy, logits


def batch_evaluate_simple(state, X_data, y_data, batch_size=1024):
    """Batch evaluation for SimpleNN."""
    all_logits, all_accs = [], []
    num_batches = (len(X_data) + batch_size - 1) // batch_size
    
    for batch_idx in range(num_batches):
        start_idx = batch_idx * batch_size
        end_idx = min(start_idx + batch_size, len(X_data))
        batch_x, batch_y = X_data[start_idx:end_idx], y_data[start_idx:end_idx]
        
        batch_acc, batch_logits = eval_step_simple(state, batch_x, batch_y)
        all_logits.append(batch_logits)
        all_accs.append(batch_acc)
    
    return np.mean(all_accs), jnp.concatenate(all_logits, axis=0)


def batch_evaluate_batchnorm(state, X_data, y_data, batch_size=1024):
    """Batch evaluation for DNN with BatchNorm."""
    all_logits, all_accs = [], []
    num_batches = (len(X_data) + batch_size - 1) // batch_size
    
    for batch_idx in range(num_batches):
        start_idx = batch_idx * batch_size
        end_idx = min(start_idx + batch_size, len(X_data))
        batch_x, batch_y = X_data[start_idx:end_idx], y_data[start_idx:end_idx]
        
        batch_acc, batch_logits = eval_step_batchnorm(state, batch_x, batch_y)
        all_logits.append(batch_logits)
        all_accs.append(batch_acc)
    
    return np.mean(all_accs), jnp.concatenate(all_logits, axis=0)


def batch_evaluate_fse(state, X_data, mask_data, y_data, batch_size=1024):
    """Batch evaluation for FSE+Attention."""
    all_logits, all_accs = [], []
    num_batches = (len(X_data) + batch_size - 1) // batch_size
    
    for batch_idx in range(num_batches):
        start_idx = batch_idx * batch_size
        end_idx = min(start_idx + batch_size, len(X_data))
        batch_x = X_data[start_idx:end_idx]
        batch_mask = mask_data[start_idx:end_idx]
        batch_y = y_data[start_idx:end_idx]
        
        batch_acc, batch_logits = eval_step_fse(state, batch_x, batch_mask, batch_y)
        all_logits.append(batch_logits)
        all_accs.append(batch_acc)
    
    return np.mean(all_accs), jnp.concatenate(all_logits, axis=0)


def batch_evaluate_fse_aware(state, X_data, mask_data, modes_data, y_data, batch_size=1024):
    """Batch evaluation for Detector-Aware FSE+Attention (Phase 1)"""
    all_logits = []
    all_accs = []
    
    num_batches = (len(X_data) + batch_size - 1) // batch_size
    
    for batch_idx in range(num_batches):
        start_idx = batch_idx * batch_size
        end_idx = min(start_idx + batch_size, len(X_data))
        
        batch_x = X_data[start_idx:end_idx]
        batch_mask = mask_data[start_idx:end_idx]
        batch_modes = modes_data[start_idx:end_idx]
        batch_y = y_data[start_idx:end_idx]
        
        batch_acc, batch_logits = eval_step_fse_aware(state, batch_x, batch_mask, 
                                                     batch_modes, batch_y)
        all_logits.append(batch_logits)
        all_accs.append(batch_acc)
    
    all_logits = jnp.concatenate(all_logits, axis=0)
    avg_acc = np.mean(all_accs)
    
    return avg_acc, all_logits


# ============================================================================
# 3.5: EXTENDED TRAINSTATE FOR BATCHNORM
# ============================================================================

class TrainStateWithBatchStats(train_state.TrainState):
    """Extended TrainState that includes batch_stats for BatchNorm."""
    batch_stats: any = None


# ============================================================================
# 3.6: UNIFIED TRAINING ORCHESTRATOR
# ============================================================================

def train_model(model_type, momentum_range, preprocessing_data, force_training=False):
    """
    Unified training function for all model types.
    
    Args:
        model_type: One of 'JAX_SimpleNN', 'JAX_DNN', 'JAX_FSE_Attention', 'JAX_FSE_Attention_DetectorAware'
        momentum_range: Momentum range dict
        preprocessing_data: Dict with preprocessed data
        force_training: If True, train from scratch; if False, try to load
    
    Returns:
        Dict with training results
    """
    mr_key = momentum_range['key']
    params = HYPERPARAMETERS[model_type]
    
    print(f"\n{'*'*80}")
    print(f"{model_type} - {momentum_range['name']}")
    print(f"{'*'*80}\n")
    
    # Try to load existing model
    if not force_training:
        loaded, _ = load_single_model(mr_key, model_type)
        if loaded is not None:
            print(f"✓ Loaded existing model (skipped training)")
            return loaded
    
    print("Training from scratch...")
    print(f"✓ Hyperparameters:")
    for k, v in params.items():
        print(f"    {k:20s}: {v}")
    
    # Get preprocessed data
    X_train = preprocessing_data['X_train_scaled']
    X_test = preprocessing_data['X_test_scaled']
    y_train = preprocessing_data['y_train']
    y_test = preprocessing_data['y_test']
    
    # Convert to JAX
    X_train_jax = jnp.array(X_train, dtype=jnp.float32)
    X_test_jax = jnp.array(X_test, dtype=jnp.float32)
    y_train_jax = jnp.array(y_train, dtype=jnp.int32)
    y_test_jax = jnp.array(y_test, dtype=jnp.int32)
    
    # Compute class weights
    class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
    class_weights_jax = jnp.array(list(dict(enumerate(class_weights)).values()), dtype=jnp.float32)
    
    print(f"\n✓ Class weights:")
    for i, w in enumerate(class_weights):
        print(f"    {PARTICLE_NAMES[i]:10s}: {w:.4f}")
    
    # Initialise model based on type
    key = random.PRNGKey(RANDOM_SEED + hash(model_type) % 10000)
    
    if model_type == 'JAX_SimpleNN':
        model = JAX_SimpleNN(
            hidden_dims=params['hidden_dims'],
            num_classes=NUM_CLASSES,
            dropout_rate=params['dropout_rate']
        )
        dummy_input = jnp.ones((1, X_train_jax.shape[1]))
        model_params = model.init(key, dummy_input, training=False)
        
        tx = optax.adam(params['learning_rate'])
        state = train_state.TrainState.create(
            apply_fn=model.apply,
            params=model_params['params'],
            tx=tx
        )
        
        train_fn = train_step_simple
        eval_fn = batch_evaluate_simple
        
    elif model_type == 'JAX_DNN':
        model = JAX_DNN(
            hidden_dims=params['hidden_dims'],
            num_classes=NUM_CLASSES,
            dropout_rate=params['dropout_rate']
        )
        dummy_input = jnp.ones((1, X_train_jax.shape[1]))
        variables = model.init(key, dummy_input, training=True)
        
        model_params = variables['params']
        batch_stats = variables.get('batch_stats', {})
        
        tx = optax.adam(params['learning_rate'])
        state = TrainStateWithBatchStats.create(
            apply_fn=model.apply,
            params=model_params,
            tx=tx,
            batch_stats=batch_stats
        )
        
        train_fn = train_step_batchnorm
        eval_fn = batch_evaluate_batchnorm
        
    elif model_type == 'JAX_FSE_Attention':
        model = JAX_FSE_Attention(
            hidden_dim=params['hidden_dim'],
            num_heads=params['num_heads'],
            num_classes=NUM_CLASSES,
            dropout_rate=params['dropout_rate']
        )
        
        masks_train_jax = jnp.array(preprocessing_data['masks_train'], dtype=jnp.float32)
        masks_test_jax = jnp.array(preprocessing_data['masks_test'], dtype=jnp.float32)
        
        dummy_input = jnp.ones((1, X_train_jax.shape[1]))
        dummy_mask = jnp.ones((1, masks_train_jax.shape[1]))
        model_params = model.init(key, dummy_input, dummy_mask, training=False)
        
        tx = optax.adam(params['learning_rate'])
        state = train_state.TrainState.create(
            apply_fn=model.apply,
            params=model_params['params'],
            tx=tx
        )
        
        train_fn = train_step_fse
        eval_fn = batch_evaluate_fse
        
    elif model_type == 'JAX_FSE_Attention_DetectorAware':
        model = JAX_FSE_Attention_DetectorAware(
            hidden_dim=params['hidden_dim'],
            num_heads=params['num_heads'],
            num_classes=NUM_CLASSES,
            dropout_rate=params['dropout_rate'],
            detector_embed_dim=8
        )
        
        masks_train_jax = jnp.array(preprocessing_data['masks_train'], dtype=jnp.float32)
        masks_test_jax = jnp.array(preprocessing_data['masks_test'], dtype=jnp.float32)
        detector_modes_train_jax = jnp.array(preprocessing_data['detector_modes_train'], dtype=jnp.int32)
        detector_modes_test_jax = jnp.array(preprocessing_data['detector_modes_test'], dtype=jnp.int32)
        
        dummy_input = jnp.ones((1, X_train_jax.shape[1]))
        dummy_mask = jnp.ones((1, masks_train_jax.shape[1]))
        dummy_modes = jnp.zeros((1,), dtype=jnp.int32)
        model_params = model.init(key, dummy_input, dummy_mask, dummy_modes, training=False)
        
        tx = optax.adam(params['learning_rate'])
        state = train_state.TrainState.create(
            apply_fn=model.apply,
            params=model_params['params'],
            tx=tx
        )
        
        train_fn = train_step_fse_aware
        eval_fn = batch_evaluate_fse_aware
    
    print(f"✓ Model initialised")
    
    # Training loop
    num_batches = len(X_train_jax) // params['batch_size']
    best_val_acc = 0.0
    patience_counter = 0
    train_losses, val_accuracies = [], []
    main_key = key
    
    print(f"\nTraining (max {params['num_epochs']} epochs, patience={params['patience']})...\n")
    
    for epoch in range(params['num_epochs']):
        main_key, shuffle_key, dropout_key = random.split(main_key, 3)
        perm = random.permutation(shuffle_key, len(X_train_jax))
        X_train_shuffled = X_train_jax[perm]
        y_train_shuffled = y_train_jax[perm]
        
        if model_type in ['JAX_FSE_Attention', 'JAX_FSE_Attention_DetectorAware']:
            masks_train_shuffled = masks_train_jax[perm]
        
        if model_type == 'JAX_FSE_Attention_DetectorAware':
            detector_modes_train_shuffled = detector_modes_train_jax[perm]
        
        epoch_losses = []
        for batch_idx in range(num_batches):
            dropout_key, subkey = random.split(dropout_key)
            start_idx = batch_idx * params['batch_size']
            end_idx = start_idx + params['batch_size']
            batch_x = X_train_shuffled[start_idx:end_idx]
            batch_y = y_train_shuffled[start_idx:end_idx]
            
            if model_type == 'JAX_FSE_Attention':
                batch_mask = masks_train_shuffled[start_idx:end_idx]
                state, loss = train_fn(state, batch_x, batch_mask, batch_y, subkey, class_weights_jax)
            elif model_type == 'JAX_FSE_Attention_DetectorAware':
                batch_mask = masks_train_shuffled[start_idx:end_idx]
                batch_modes = detector_modes_train_shuffled[start_idx:end_idx]
                state, loss = train_fn(state, batch_x, batch_mask, batch_modes, batch_y, subkey, class_weights_jax)
            else:
                state, loss = train_fn(state, batch_x, batch_y, subkey, class_weights_jax)
            
            epoch_losses.append(loss)
        
        avg_train_loss = np.mean(epoch_losses)
        train_losses.append(avg_train_loss)
        
        # Validation
        if model_type == 'JAX_FSE_Attention':
            val_acc, _ = eval_fn(state, X_test_jax, masks_test_jax, y_test_jax, batch_size=1024)
        elif model_type == 'JAX_FSE_Attention_DetectorAware':
            val_acc, _ = eval_fn(state, X_test_jax, masks_test_jax, detector_modes_test_jax, y_test_jax, batch_size=1024)
        else:
            val_acc, _ = eval_fn(state, X_test_jax, y_test_jax, batch_size=1024)
        
        val_accuracies.append(float(val_acc))
        
        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"Epoch {epoch+1:3d}/{params['num_epochs']} | Loss: {avg_train_loss:.4f} | Val Acc: {val_acc:.4f}")
        
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            patience_counter = 0
            best_params = state.params
            if model_type == 'JAX_DNN':
                best_batch_stats = state.batch_stats
        else:
            patience_counter += 1
            if patience_counter >= params['patience']:
                print(f"✓ Early stopping at epoch {epoch+1}")
                break
    
    # Restore best parameters
    state = state.replace(params=best_params)
    if model_type == 'JAX_DNN':
        state = state.replace(batch_stats=best_batch_stats)
    
    # Final evaluation
    print(f"\nFinal evaluation...")
    if model_type == 'JAX_FSE_Attention':
        train_acc, train_logits = eval_fn(state, X_train_jax, masks_train_jax, y_train_jax, batch_size=1024)
        test_acc, test_logits = eval_fn(state, X_test_jax, masks_test_jax, y_test_jax, batch_size=1024)
    elif model_type == 'JAX_FSE_Attention_DetectorAware':
        train_acc, train_logits = eval_fn(state, X_train_jax, masks_train_jax, detector_modes_train_jax, y_train_jax, batch_size=1024)
        test_acc, test_logits = eval_fn(state, X_test_jax, masks_test_jax, detector_modes_test_jax, y_test_jax, batch_size=1024)
    else:
        train_acc, train_logits = eval_fn(state, X_train_jax, y_train_jax, batch_size=1024)
        test_acc, test_logits = eval_fn(state, X_test_jax, y_test_jax, batch_size=1024)
    
    train_probs = jax.nn.softmax(train_logits, axis=-1)
    test_probs = jax.nn.softmax(test_logits, axis=-1)
    y_pred_test = jnp.argmax(test_logits, axis=-1)
    
    print(f"\n✓ Results:")
    print(f"    Train Acc:    {train_acc:.4f}")
    print(f"    Test Acc:     {test_acc:.4f}")
    print(f"    Best Val Acc: {best_val_acc:.4f}")
    
    # Store results
    results = {
        'model_type': model_type,
        'hyperparameters': params,
        'train_losses': train_losses,
        'val_accuracies': val_accuracies,
        'best_val_acc': float(best_val_acc),
        'train_acc': float(train_acc),
        'test_acc': float(test_acc),
        'train_probs': train_probs,
        'test_probs': test_probs,
        'y_pred_test': y_pred_test,
        'y_test': y_test_jax,
    }
    
    # Save model
    save_single_model(mr_key, model_type, results)
    
    return results


print("✓ Focal loss function defined")
print("✓ Model architectures defined (SimpleNN, DNN, FSE+Attention, FSE+Attention_DetectorAware)")
print("✓ Training step functions defined (JIT compiled, including Phase 1)")
print("✓ Evaluation functions defined (batch-aware, including Phase 1)")
print("✓ Extended TrainState defined for BatchNorm")
print("✓ Unified training orchestrator defined (supports all model types)")
print(f"\n{'='*80}")
print("✓ SECTION 3 COMPLETE: All training utilities ready (Phase 1 integrated)")
print(f"{'='*80}\n")


## Section 4: Data Loading & Initialisation

In [ ]:
# ============================================================================
# SECTION 4.0: DATA LOADING & INITIALISATION
# ============================================================================

# Load data once
df = load_data(CSV_PATH)

# Initialise master results storage
all_results_by_model_and_range = {}

print("✓ Data loaded")
print("✓ Results storage initialised")
print(f"\n{'='*80}")
print("✓ SECTION 4.0 COMPLETE: Ready for training")
print(f"{'='*80}\n")



### Section 4A: Train JAX_SimpleNN

In [ ]:
# ============================================================================
# SECTION 4A: TRAIN JAX_SIMPLENN MODEL (REFACTORED)
# ============================================================================

print(f"\n{'#'*80}")
print("SECTION 4A: TRAINING JAX_SIMPLENN")
print(f"{'#'*80}\n")

# Train SimpleNN for all momentum ranges
for momentum_range in MOMENTUM_RANGES:
    mr_key = momentum_range['key']
    
    print(f"\n{'='*80}")
    print(f"MOMENTUM RANGE: {momentum_range['name']}")
    print(f"{'='*80}\n")
    
    # Get or create preprocessing data
    if mr_key not in all_results_by_model_and_range:
        # First model for this range - preprocess data
        preprocessing_data = preprocess_momentum_range(df, momentum_range)
        all_results_by_model_and_range[mr_key] = {
            'preprocessing': preprocessing_data,
            'models': {}
        }
    else:
        # Reuse existing preprocessing
        preprocessing_data = all_results_by_model_and_range[mr_key]['preprocessing']
    
    # Train/load SimpleNN
    force_training = FORCE_TRAINING['JAX_SimpleNN'][mr_key]
    
    results = train_model(
        model_type='JAX_SimpleNN',
        momentum_range=momentum_range,
        preprocessing_data=preprocessing_data,
        force_training=force_training
    )
    
    all_results_by_model_and_range[mr_key]['models']['JAX_SimpleNN'] = results

print(f"\n{'='*80}")
print("✓ SECTION 4A COMPLETE: JAX_SimpleNN trained/loaded for all ranges")
print(f"{'='*80}\n")

# Summary for SimpleNN
print("\nJAX_SimpleNN Summary:")
for momentum_range in MOMENTUM_RANGES:
    mr_key = momentum_range['key']
    results = all_results_by_model_and_range[mr_key]['models']['JAX_SimpleNN']
    print(f"  {momentum_range['name']:30s}: Test Acc = {results['test_acc']:.4f}")
    

### Section 4B: Train JAX_DNN

In [ ]:
# ============================================================================
# SECTION 4B: TRAIN JAX_DNN MODEL
# ============================================================================

# Train DNN for all momentum ranges
for momentum_range in MOMENTUM_RANGES:
    mr_key = momentum_range['key']
    
    print(f"\n{'='*80}")
    print(f"MOMENTUM RANGE: {momentum_range['name']}")
    print(f"{'='*80}\n")
    
    # Get or create preprocessing data
    if mr_key not in all_results_by_model_and_range:
        # First model for this range - preprocess data
        preprocessing_data = preprocess_momentum_range(df, momentum_range)
        all_results_by_model_and_range[mr_key] = {
            'preprocessing': preprocessing_data,
            'models': {}
        }
    else:
        # Reuse existing preprocessing
        preprocessing_data = all_results_by_model_and_range[mr_key]['preprocessing']
    
    # Train/load DNN
    force_training = FORCE_TRAINING['JAX_DNN'][mr_key]
    
    results = train_model(
        model_type='JAX_DNN',
        momentum_range=momentum_range,
        preprocessing_data=preprocessing_data,
        force_training=force_training
    )
    
    all_results_by_model_and_range[mr_key]['models']['JAX_DNN'] = results

print(f"\n{'='*80}")
print("✓ SECTION 4B COMPLETE: JAX_DNN trained/loaded for all ranges")
print(f"{'='*80}\n")

# Summary for DNN
print("\nJAX_DNN Summary:")
for momentum_range in MOMENTUM_RANGES:
    mr_key = momentum_range['key']
    results = all_results_by_model_and_range[mr_key]['models']['JAX_DNN']
    print(f"  {momentum_range['name']:30s}: Test Acc = {results['test_acc']:.4f}")
    

### Section 4C: Train JAX_FSE_Attention & JAX_FSE_Detector_Aware

In [ ]:
# ============================================================================
# SECTION 4C: TRAIN JAX_FSE_ATTENTION & JAX_FSE_ATTENTION_DETECTORAWARE
# ============================================================================

# ============================================================================
# 4C.1: TRAIN STANDARD JAX_FSE_ATTENTION
# ============================================================================

print(f"\n{'*'*80}")
print("PHASE 0: STANDARD JAX_FSE_ATTENTION (baseline)")
print(f"{'*'*80}\n")

for momentum_range in MOMENTUM_RANGES:
    mr_key = momentum_range['key']
    
    print(f"\n{'='*80}")
    print(f"MOMENTUM RANGE: {momentum_range['name']}")
    print(f"{'='*80}\n")
    
    # Get or create preprocessing data
    if mr_key not in all_results_by_model_and_range:
        # First model for this range - preprocess data
        preprocessing_data = preprocess_momentum_range(df, momentum_range)
        all_results_by_model_and_range[mr_key] = {
            'preprocessing': preprocessing_data,
            'models': {}
        }
    else:
        # Reuse existing preprocessing
        preprocessing_data = all_results_by_model_and_range[mr_key]['preprocessing']
    
    # Train/load FSE+Attention
    force_training = FORCE_TRAINING['JAX_FSE_Attention'][mr_key]
    
    results = train_model(
        model_type='JAX_FSE_Attention',
        momentum_range=momentum_range,
        preprocessing_data=preprocessing_data,
        force_training=force_training
    )
    
    all_results_by_model_and_range[mr_key]['models']['JAX_FSE_Attention'] = results

print(f"\n{'='*80}")
print("✓ PHASE 0 COMPLETE: JAX_FSE_Attention trained/loaded for all ranges")
print(f"{'='*80}\n")

# Summary for FSE+Attention
print("\nJAX_FSE_Attention Summary (Standard FSE baseline):")
for momentum_range in MOMENTUM_RANGES:
    mr_key = momentum_range['key']
    results = all_results_by_model_and_range[mr_key]['models']['JAX_FSE_Attention']
    print(f"  {momentum_range['name']:30s}: Test Acc = {results['test_acc']:.4f}")

# ============================================================================
# 4C.2: TRAIN PHASE 1 JAX_FSE_ATTENTION_DETECTORAWARE
# ============================================================================

print(f"\n{'*'*80}")
print("PHASE 1: JAX_FSE_ATTENTION_DETECTORAWARE (detector-aware, improved)")
print(f"{'*'*80}\n")

# Add Phase 1 flag if not already present
if 'JAX_FSE_Attention_DetectorAware' not in FORCE_TRAINING:
    FORCE_TRAINING['JAX_FSE_Attention_DetectorAware'] = {
        'full': False,
        '0.7-1.5': False,
        '1-3': False
    }

for momentum_range in MOMENTUM_RANGES:
    mr_key = momentum_range['key']
    
    print(f"\n{'='*80}")
    print(f"MOMENTUM RANGE: {momentum_range['name']} (Detector-Aware FSE)")
    print(f"{'='*80}\n")
    
    # Get preprocessing data (already created in Phase 0)
    preprocessing_data = all_results_by_model_and_range[mr_key]['preprocessing']
    
    # Train/load Detector-Aware FSE
    force_training = FORCE_TRAINING['JAX_FSE_Attention_DetectorAware'][mr_key]
    
    results = train_model(
        model_type='JAX_FSE_Attention_DetectorAware',
        momentum_range=momentum_range,
        preprocessing_data=preprocessing_data,
        force_training=force_training
    )
    
    all_results_by_model_and_range[mr_key]['models']['JAX_FSE_Attention_DetectorAware'] = results

print(f"\n{'='*80}")
print("✓ PHASE 1 COMPLETE: JAX_FSE_Attention_DetectorAware trained/loaded for all ranges")
print(f"{'='*80}\n")

# Summary for Detector-Aware FSE
print("\nJAX_FSE_Attention_DetectorAware Summary (Phase 1 detector-aware):")
for momentum_range in MOMENTUM_RANGES:
    mr_key = momentum_range['key']
    results = all_results_by_model_and_range[mr_key]['models']['JAX_FSE_Attention_DetectorAware']
    print(f"  {momentum_range['name']:30s}: Test Acc = {results['test_acc']:.4f}")

# ============================================================================
# 4C.3: COMPARISON - STANDARD VS DETECTOR-AWARE FSE
# ============================================================================

print(f"\n{'='*80}")
print("PHASE 1 vs PHASE 0: IMPROVEMENT SUMMARY")
print(f"{'='*80}\n")

for momentum_range in MOMENTUM_RANGES:
    mr_key = momentum_range['key']
    
    std_results = all_results_by_model_and_range[mr_key]['models']['JAX_FSE_Attention']
    aware_results = all_results_by_model_and_range[mr_key]['models']['JAX_FSE_Attention_DetectorAware']
    
    std_acc = std_results['test_acc']
    aware_acc = aware_results['test_acc']
    improvement = aware_acc - std_acc
    improvement_pct = (improvement / std_acc * 100) if std_acc > 0 else 0
    
    print(f"{momentum_range['name']:30s}:")
    print(f"  Standard FSE:       {std_acc:.4f}")
    print(f"  Detector-Aware FSE: {aware_acc:.4f}")
    print(f"  Improvement:        {improvement:+.4f} ({improvement_pct:+.2f}%)")
    print()

print(f"{'='*80}")
print("✓ SECTION 4C COMPLETE: Both standard and detector-aware FSE trained")
print(f"{'='*80}\n")
